In [25]:
import json
import floorplan_backend
import time

In [ ]:
with open('./samples/topology.json') as topology_json:
    with open('./samples/construction.json') as construction_json:
        topology = json.load(topology_json)
        construction = json.load(construction_json)
        start = time.time()
        building = floorplan_backend.Building.from_data(topology, construction)
        print(time.time()-start)
        building.solve_joints()
        geoms = {
            uuid: result
            for face_id in building.face_ids()
            for uuid, result in building.layer_geometry(face_id).items()
        }

0.005057811737060547


In [27]:
import numpy as np

def build_joint_graph(construction, geoms):
    graph = {"nodes": {}, "edges": {}}

    # Nodes: materials and composites
    for node_type in ("materials", "composites"):
        for uuid, data in construction.get(node_type, {}).items():
            graph["nodes"][uuid] = {
                "kind": node_type[:-1],
                **data,
                "area": 0.0,
                "volume": 0.0,
            }

    # Aggregate layer area and volume by material
    for face_layers in geoms.values():
        for layer_geom in face_layers:
            layer_data = construction["layers"][layer_geom["uuid"]]
            material_uuid = layer_data["material"]

            area = float(layer_geom["area"])
            thickness = float(layer_data["thickness"])
            volume = area * thickness

            if material_uuid not in graph["nodes"]:
                continue

            graph["nodes"][material_uuid]["area"] += area
            graph["nodes"][material_uuid]["volume"] += volume

    # Aggregate area and volume into composites using component fractions
    for uuid, node in graph["nodes"].items():
        if node["kind"] != "composite":
            continue

        for part, fraction_key in (
            ("part1", "part1_fraction"),
            ("part2", "part2_fraction"),
        ):
            material_uuid = node.get(part)
            fraction = float(node.get(fraction_key, 0.0))

            if material_uuid not in graph["nodes"]:
                continue

            material = graph["nodes"][material_uuid]
            node["area"] += material["area"] * fraction
            node["volume"] += material["volume"] * fraction

    # Edges: material pairs containing all face-pair joints
    for face_uuid, face_layers in geoms.items():
        for layer_geom in face_layers:
            layer_data = construction["layers"][layer_geom["uuid"]]
            material_uuid = layer_data["material"]

            points = np.asarray(layer_geom["inner"])
            lengths = np.linalg.norm(
                np.roll(points, -1, axis=0) - points,
                axis=1,
            )

            for edge_idx, joint in enumerate(layer_geom["joints"]):
                if joint is None:
                    continue

                other_face_uuid = joint["face_Id"]

                # Process each face pair once
                if face_uuid > other_face_uuid:
                    continue

                other_layer = geoms[other_face_uuid][joint["layer_idx"]]
                other_material_uuid = construction["layers"][
                    other_layer["uuid"]
                ]["material"]

                # Edge identity: sorted material pair
                material_pair = tuple(sorted((
                    material_uuid,
                    other_material_uuid,
                )))
                edge_key = "|".join(material_pair)

                # Joint identity: sorted face pair
                face_pair = tuple(sorted((
                    face_uuid,
                    other_face_uuid,
                )))
                face_key = "|".join(face_pair)

                edge = graph["edges"].setdefault(
                    edge_key,
                    {
                        "source": material_pair[0],
                        "target": material_pair[1],
                        "joints": {},
                    },
                )

                edge["joints"][face_key] = float(lengths[edge_idx])

    return graph


graph = build_joint_graph(construction, geoms)

In [28]:
import numpy as np
import plotly.graph_objects as go
import networkx as nx

import numpy as np
import plotly.graph_objects as go
import networkx as nx


def plot_joint_graph(joint_graph, output="joint_graph.html"):
    G = nx.Graph()

    # Add nodes with technical properties, area and volume
    for uuid, data in joint_graph["nodes"].items():
        G.add_node(uuid, **data)

    # Add joint edges, including self-connections
    for edge in joint_graph["edges"].values():
        joints = edge["joints"]
        source = edge["source"]
        target = edge["target"]
        if not source==target:
            G.add_edge(
                source,
                target,
                total_length=sum(joints.values()),
                joint_count=len(joints),
                joints=joints,
            )

    # Connect composites to their constituent materials
    membership_edges = []
    for composite, data in G.nodes(data=True):
        if data.get("kind") != "composite":
            continue

        for part in ("part1", "part2"):
            material = data.get(part)
            if material in G:
                membership_edges.append((composite, material))

    # Compact 3D layout
    layout_graph = G.copy()
    layout_graph.add_edges_from(membership_edges)

    pos = nx.spring_layout(
        layout_graph,
        dim=3,
        seed=42,
        k=0.25,
        iterations=100,
        scale=0.7,
    )

    # Dotted composite-to-material links
    mx, my, mz = [], [], []

    for u, v in membership_edges:
        mx.extend([pos[u][0], pos[v][0], None])
        my.extend([pos[u][1], pos[v][1], None])
        mz.extend([pos[u][2], pos[v][2], None])

    membership_trace = go.Scatter3d(
        x=mx,
        y=my,
        z=mz,
        mode="lines",
        line=dict(color="gray", width=3, dash="dot"),
        hoverinfo="skip",
        name="Composite membership",
    )

    # Joint edge styling
    edges = list(G.edges(data=True))

    max_length = max(
        (d["total_length"] for _, _, d in edges), default=1
    ) or 1

    max_count = max(
        (d["joint_count"] for _, _, d in edges), default=1
    ) or 1

    edge_traces = []

    # Loop geometry settings
    loop_radius = 0.12
    loop_points = 60

    for u, v, data in edges:
        width = 1 + (100 * data["total_length"] / max_length)**0.5
        edge_color = data["joint_count"]

        if u == v:
            # Self-connection: draw a 3D loop around its node
            center = np.asarray(pos[u], dtype=float)
            t = np.linspace(0, 2 * np.pi, loop_points)

            x = center[0] + loop_radius * np.cos(t)
            y = center[1] + loop_radius * np.sin(t)
            z = center[2] + loop_radius * 0.35 * np.sin(2 * t)

        else:
            # Normal connection between two nodes
            x = [pos[u][0], pos[v][0]]
            y = [pos[u][1], pos[v][1]]
            z = [pos[u][2], pos[v][2]]

        node_u = G.nodes[u]
        node_v = G.nodes[v]

        average_length = (
            data["total_length"] / data["joint_count"]
            if data["joint_count"] else 0
        )

        hover_text = (
            f"<b>{node_u.get('name', u)} — "
            f"{node_v.get('name', v)}</b>"
            f"<br>Connection type: "
            f"{'Self-connection' if u == v else 'Joint'}"
            f"<br>Number of connections: {data['joint_count']}"
            f"<br>Total joint length: {data['total_length']:.3f}"
            f"<br>Average joint length: {average_length:.3f}"
        )

        edge_traces.append(go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode="lines",
            line=dict(
                width=width,
                color=[edge_color] * len(x),
                colorscale="Viridis",
                cmin=0,
                cmax=max_count,
            ),
            hovertemplate=hover_text + "<extra></extra>",
            showlegend=False,
        ))

    # Scale node sizes by volume, separately for each node type
    nodes = list(G.nodes)

    material_volumes = {
        n: max(0.0, float(G.nodes[n].get("volume", 0.0)))
        for n in nodes
        if G.nodes[n].get("kind") == "material"
    }

    composite_volumes = {
        n: max(0.0, float(G.nodes[n].get("volume", 0.0)))
        for n in nodes
        if G.nodes[n].get("kind") == "composite"
    }

    max_material_volume = max(material_volumes.values(), default=0.0)
    max_composite_volume = max(composite_volumes.values(), default=0.0)

    node_sizes = []

    for n in nodes:
        data = G.nodes[n]
        volume = max(0.0, float(data.get("volume", 0.0)))

        if data.get("kind") == "material":
            max_volume = max_material_volume
        else:
            max_volume = max_composite_volume

        size = (
            7 + 100 * volume / max_volume
            if max_volume > 0 else 7
        )
        node_sizes.append(size)

    # Node hover information
    node_text = []

    for n in nodes:
        data = G.nodes[n]
        name = data.get("name", n)

        properties = "<br>".join(
            f"{key}: {value}"
            for key, value in data.items()
            if key not in (
                "name",
                "kind",
                "part1",
                "part2",
                "part1_fraction",
                "part2_fraction",
            )
        )

        node_text.append(
            f"<b>{name}</b>"
            f"<br>UUID: {n}"
            f"<br>Type: {data.get('kind', '')}"
            + (f"<br>{properties}" if properties else "")
        )

    node_trace = go.Scatter3d(
        x=[pos[n][0] for n in nodes],
        y=[pos[n][1] for n in nodes],
        z=[pos[n][2] for n in nodes],
        mode="markers+text",
        text=[G.nodes[n].get("name", n) for n in nodes],
        textposition="top center",
        customdata=node_text,
        
        hovertemplate="%{customdata}<extra></extra>",
        marker=dict(
            size=node_sizes,
            color=[
                "#2878B5"
                if G.nodes[n].get("kind") == "material"
                else "#E58B32"
                for n in nodes
            ],
            sizemode="diameter",
        ),
        name="Materials and composites",
        showlegend=False,
    )

    # Legend entries for node types, membership and joint length
    legend_traces = [
        go.Scatter3d(
            x=[None], y=[None], z=[None],
            mode="markers",
            marker=dict(size=8, color="#2878B5"),
            name="Material",
            hoverinfo="skip",
        ),
        go.Scatter3d(
            x=[None], y=[None], z=[None],
            mode="markers",
            marker=dict(size=8, color="#E58B32"),
            name="Composite",
            hoverinfo="skip",
        ),

        go.Scatter3d(
            x=[None], y=[None], z=[None],
            mode="lines",
            line=dict(color="gray", width=2),
            name="Short joint length",
            hoverinfo="skip",
        ),
        go.Scatter3d(
            x=[None], y=[None], z=[None],
            mode="lines",
            line=dict(color="gray", width=10),
            name="Long joint length",
            hoverinfo="skip",
        )
    ]

    # Continuous colour scale for number of connections
    color_scale_trace = go.Scatter3d(
        x=[None, None],
        y=[None, None],
        z=[None, None],
        mode="markers",
        marker=dict(
            size=0,
            color=[0, max_count],
            colorscale="Viridis",
            cmin=0,
            cmax=max_count,
            showscale=True,
            colorbar=dict(
                title=dict(text="Number of connections"),
                x=1.02,
                y=0.5,
                len=0.65,
                thickness=18,
            ),
        ),
        hoverinfo="skip",
        showlegend=False,
    )

    # Assemble figure
    fig = go.Figure(
        data=[
            membership_trace,
            *edge_traces,
            node_trace,
            *legend_traces,
            color_scale_trace,
        ]
    )

    fig.update_layout(
        title="Construction Joint Graph",
        showlegend=True,
        legend=dict(
            title="Graph legend",
            x=1.02,
            y=1,
        ),
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode="data",
        ),
        margin=dict(l=0, r=220, t=40, b=0),
    )

    fig.write_html(output, include_plotlyjs=True)
    fig.show()

    return fig


fig = plot_joint_graph(graph)
fig = plot_joint_graph(graph)

